# Analyzing Historical Stock/Revenue Data and Building a Dashboard
Tesla (TSLA) and GameStop (GME): share price from `yfinance`, revenue from web scraping.

In [1]:
!pip install yfinance beautifulsoup4 lxml requests plotly nbformat

In [2]:
import io
import requests
import pandas as pd
import yfinance as yf
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from bs4 import BeautifulSoup

import warnings
warnings.filterwarnings("ignore", category=FutureWarning)

### Helper: dashboard function

In [3]:
def make_graph(stock_data, revenue_data, stock):
    fig = make_subplots(rows=2, cols=1, shared_xaxes=True,
                        subplot_titles=("Historical Share Price", "Historical Revenue"),
                        vertical_spacing=.3)
    stock_data = stock_data.copy()
    revenue_data = revenue_data.copy()
    stock_data["Date"] = pd.to_datetime(stock_data["Date"]).dt.tz_localize(None)
    revenue_data["Date"] = pd.to_datetime(revenue_data["Date"])
    stock_data_specific = stock_data[stock_data["Date"] <= "2021-06-14"]
    revenue_data_specific = revenue_data[revenue_data["Date"] <= "2021-04-30"]
    fig.add_trace(go.Scatter(x=stock_data_specific["Date"],
                             y=stock_data_specific["Close"].astype("float"),
                             name="Share Price"), row=1, col=1)
    fig.add_trace(go.Scatter(x=revenue_data_specific["Date"],
                             y=revenue_data_specific["Revenue"].astype("float"),
                             name="Revenue"), row=2, col=1)
    fig.update_xaxes(title_text="Date", row=1, col=1)
    fig.update_xaxes(title_text="Date", row=2, col=1)
    fig.update_yaxes(title_text="Price ($US)", row=1, col=1)
    fig.update_yaxes(title_text="Revenue ($US Millions)", row=2, col=1)
    fig.update_layout(showlegend=False, height=900, title=stock, xaxis_rangeslider_visible=True)
    fig.show()


def scrape_revenue(url):
    """Download a page and return its revenue table as Date / Revenue columns."""
    html = requests.get(url, timeout=30).text
    soup = BeautifulSoup(html, "html.parser")
    for table in soup.find_all("table"):
        df = pd.read_html(io.StringIO(str(table)))[0]
        if any("revenue" in str(c).lower() for c in df.columns):
            df = df.iloc[:, :2]
            df.columns = ["Date", "Revenue"]
            df["Revenue"] = df["Revenue"].astype(str).str.replace(r"[$,]", "", regex=True)
            df = df[df["Revenue"].str.strip() != ""]
            df = df[df["Revenue"] != "nan"].dropna().reset_index(drop=True)
            return df
    raise ValueError("No revenue table found at " + url)

## Question 1 - Extracting Tesla Stock Data Using yfinance

In [4]:
tesla = yf.Ticker("TSLA")
tesla_data = tesla.history(period="max")
tesla_data.reset_index(inplace=True)
tesla_data.head()

,Date,Open,High,Low,Close,Volume,Dividends,Stock Splits
0,2010-06-29 00:00:00-04:00,1.266667,1.666667,1.169333,1.592667,281494500,0.0,0.0
1,2010-06-30 00:00:00-04:00,1.719333,2.028000,1.553333,1.588667,257806500,0.0,0.0
2,2010-07-01 00:00:00-04:00,1.666667,1.728000,1.351333,1.464000,123282000,0.0,0.0
3,2010-07-02 00:00:00-04:00,1.533333,1.540000,1.247333,1.280000,77097000,0.0,0.0
4,2010-07-06 00:00:00-04:00,1.333333,1.333333,1.055333,1.074000,103003500,0.0,0.0


## Question 2 - Extracting Tesla Revenue Data Using Webscraping

In [5]:
url = ("https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/"
       "IBMDeveloperSkillsNetwork-PY0220EN-SkillsNetwork/labs/project/revenue.htm")
tesla_revenue = scrape_revenue(url)
tesla_revenue.tail()

,Date,Revenue
8,2013,2013
9,2012,413
10,2011,204
11,2010,117
12,2009,112


## Question 3 - Extracting GameStop Stock Data Using yfinance

In [6]:
gme = yf.Ticker("GME")
gme_data = gme.history(period="max")
gme_data.reset_index(inplace=True)
gme_data.head()

,Date,Open,High,Low,Close,Volume,Dividends,Stock Splits
0,2002-02-13 00:00:00-05:00,1.620128,1.693350,1.603296,1.691666,76216000,0.0,0.0
1,2002-02-14 00:00:00-05:00,1.712707,1.716074,1.670626,1.683250,11021600,0.0,0.0
2,2002-02-15 00:00:00-05:00,1.683251,1.687459,1.658002,1.674834,8389600,0.0,0.0
3,2002-02-19 00:00:00-05:00,1.666418,1.666418,1.578047,1.607504,7410400,0.0,0.0
4,2002-02-20 00:00:00-05:00,1.615920,1.662210,1.603296,1.662210,6892800,0.0,0.0


## Question 4 - Extracting GameStop Revenue Data Using Webscraping

In [7]:
url = ("https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/"
       "IBMDeveloperSkillsNetwork-PY0220EN-SkillsNetwork/labs/project/stock.html")
gme_revenue = scrape_revenue(url)
gme_revenue.tail()

,Date,Revenue
11,2009,8806
12,2008,7094
13,2007,5319
14,2006,3092
15,2005,1843


## Question 5 - Tesla Stock and Revenue Dashboard

In [8]:
make_graph(tesla_data, tesla_revenue, "Tesla")

## Question 6 - GameStop Stock and Revenue Dashboard

In [9]:
make_graph(gme_data, gme_revenue, "GameStop")